# Understand the `static_0` table — beginner experiment

This notebook answers these basic questions before we build a model:

1. What is `static_0`?
2. Why is it stored in two files?
3. How many rows and columns does it have?
4. What does one row represent?
5. What do the unusual column names mean?
6. Does every application have a `static_0` row?
7. Do we need all the columns for our first model?

We will not train a model or perform complicated cleaning here.

## 1. What is `static_0`?

`static_0` contains internal information directly connected to the current loan application. It is a **depth-0 table**, so each `case_id` should appear once. Some columns are already-calculated summaries of earlier activity, such as behavior during the last 24 months; `static` describes the one-row-per-case shape, not an absence of historical information.

One row means **one loan application**, not necessarily one unique person. The table does not contain `target`; we get the known default outcome from `train_base` after joining on `case_id`.

In [1]:
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

project_dir = Path(r"D:\Home Credit - Credit Risk Model Stability")
train_dir = project_dir / "data" / "parquet_files" / "train"

static_paths = [
    train_dir / "train_static_0_0.parquet",
    train_dir / "train_static_0_1.parquet",
]
base_path = train_dir / "train_base.parquet"
definitions_path = project_dir / "data" / "feature_definitions.csv"

assert all(path.exists() for path in [*static_paths, base_path, definitions_path])
print("All required files were found.")

All required files were found.


## 2. Why are there two files?

The table was too large for one convenient competition file, so Kaggle divided its rows into two parts. The `_0` and `_1` at the end are file-part numbers. They are not different kinds of features.

Parquet metadata lets us count rows and columns without loading all values into memory.

In [2]:
schemas = [pq.read_schema(path) for path in static_paths]
same_schema = schemas[0].equals(schemas[1])

file_summary = pd.DataFrame([
    {
        "file": path.name,
        "rows": pq.ParquetFile(path).metadata.num_rows,
        "columns": len(schema),
        "file_size_MB": path.stat().st_size / 1_000_000,
    }
    for path, schema in zip(static_paths, schemas)
])

display(file_summary.style.format({
    "rows": "{:,}",
    "file_size_MB": "{:.1f}",
}))

print(f"Do both files have the same columns? {same_schema}")
print(f"Total rows: {file_summary['rows'].sum():,}")
print(f"Total columns: {file_summary['columns'].iloc[0]:,}")
print("The 168 columns consist of case_id plus 167 possible predictor columns.")

,file,rows,columns,file_size_MB
0,train_static_0_0.parquet,"1,003,757",168,113.9
1,train_static_0_1.parquet,"522,902",168,74.7


Do both files have the same columns? True
Total rows: 1,526,659
Total columns: 168
The 168 columns consist of case_id plus 167 possible predictor columns.


## 3. What types of columns are available?

The Parquet schema records how values are stored. A `double` is a numerical column, `string` contains text or a date stored as text, and `bool` contains True/False information.

In [3]:
schema = schemas[0]
storage_type_summary = (
    pd.Series([str(field.type) for field in schema], name="number_of_columns")
    .value_counts()
    .rename_axis("Parquet_storage_type")
    .to_frame()
)
storage_type_summary

,count
Parquet_storage_type,
double,128
string,33
bool,6
int64,1


The table contains:

- 128 numerical columns;
- 33 string columns, including 15 date columns;
- 6 Boolean columns; and
- 1 integer identifier: `case_id`.

The final letter in a feature name is a competition naming convention: `A` commonly identifies amount features, `D` date features, `M` masked categories, `P` transformed days-past-due features, and `L` other transformed features. Always read the supplied description and inspect the actual data type instead of relying only on the final letter.

## 4. What do the columns actually mean?

The file `feature_definitions.csv` is the data dictionary. We combine it with the `static_0` schema to make a simple guide for all 168 columns: `case_id` plus 167 predictor columns.

In [4]:
definitions = pd.read_csv(definitions_path)
description_map = definitions.set_index("Variable")["Description"].to_dict()

guide_rows = []
for field in schema:
    if field.name == "case_id":
        analysis_type = "identifier / join key"
        meaning = "Unique application or credit-case identifier used to join tables."
    elif field.name.endswith("D"):
        analysis_type = "date (stored as text)"
    elif pa.types.is_floating(field.type):
        analysis_type = "numerical"
    elif pa.types.is_boolean(field.type):
        analysis_type = "Boolean (True/False)"
    else:
        analysis_type = "categorical/text"

    if field.name != "case_id":
        meaning = description_map.get(field.name, "Description unavailable")

    guide_rows.append({
        "feature": field.name,
        "meaning": meaning,
        "analysis_type": analysis_type,
        "storage_type": str(field.type),
    })

column_guide = pd.DataFrame(guide_rows)
print(f"Columns in guide: {len(column_guide)}")
print(f"Columns without a meaning: {(column_guide['meaning'] == 'Description unavailable').sum()}")

guide_path = project_dir / "reports" / "static_0_all_columns_and_meanings.csv"
guide_path.parent.mkdir(parents=True, exist_ok=True)
column_guide.to_csv(guide_path, index=False)
print(f"Complete guide saved to: {guide_path}")

with pd.option_context("display.max_rows", None, "display.max_colwidth", 120):
    display(column_guide)

Columns in guide: 168
Columns without a meaning: 0
Complete guide saved to: D:\Home Credit - Credit Risk Model Stability\reports\static_0_all_columns_and_meanings.csv


,feature,meaning,analysis_type,storage_type
0,case_id,Unique application or credit-case identifier used to join tables.,identifier / join key,int64
1,actualdpdtolerance_344P,DPD of client with tolerance.,numerical,double
2,amtinstpaidbefduel24m_4187115A,Number of instalments paid before due date in the last 24 months.,numerical,double
3,annuity_780A,Monthly annuity amount.,numerical,double
4,annuitynextmonth_57A,Next month's amount of annuity.,numerical,double
5,applicationcnt_361L,Number of applications associated with the same email address as the client.,numerical,double
6,applications30d_658L,Number of applications made by the client in the last 30 days.,numerical,double
7,applicationscnt_1086L,Number of applications associated with the same phone number.,numerical,double
8,applicationscnt_464L,Number of applications made in the last 30 days by other clients with the same employer as the applicant.,numerical,double
9,applicationscnt_629L,Number of applications with the same employer in the last 7 days.,numerical,double


### Search the feature guide

You do not need to memorize 167 names. Search the feature name and description for a word such as `income`, `debt`, `application`, `payment`, or `late`.

In [5]:
def search_features(search_text):
    search_text = str(search_text)
    matches = (
        column_guide["feature"].str.contains(search_text, case=False, na=False)
        | column_guide["meaning"].str.contains(search_text, case=False, na=False)
    )
    return column_guide.loc[matches].reset_index(drop=True)

search_features("debt")

,feature,meaning,analysis_type,storage_type
0,currdebt_22A,Current debt amount of the client.,numerical,double
1,currdebtcredtyperange_828A,Current amount of debt of the applicant.,numerical,double
2,maxdebt4_972A,Maximal principal debt of the client in the hi...,numerical,double
3,totaldebt_9A,Total amount of debt.,numerical,double


## 5. Examine understandable example columns

For our first experiment, we select ten understandable numerical features. This is a learning choice, not a claim that they are the ten best features.

In [6]:
selected_features = [
    "credamount_770A",
    "annuity_780A",
    "maininc_215A",
    "applications30d_658L",
    "numactivecreds_622L",
    "totaldebt_9A",
    "avgdbddpdlast24m_3658932P",
    "avgdpdtolclosure24_3658938P",
    "clientscnt12m_3712952L",
    "price_1097A",
]

column_guide.set_index("feature").loc[selected_features]

,meaning,analysis_type,storage_type
feature,,,
credamount_770A,Loan amount or credit card limit.,numerical,double
annuity_780A,Monthly annuity amount.,numerical,double
maininc_215A,Client's primary income amount.,numerical,double
applications30d_658L,Number of applications made by the client in t...,numerical,double
numactivecreds_622L,Number of active credits.,numerical,double
totaldebt_9A,Total amount of debt.,numerical,double
avgdbddpdlast24m_3658932P,Average days past or before due of payment dur...,numerical,double
avgdpdtolclosure24_3658938P,Average DPD (days past due) with tolerance wit...,numerical,double
clientscnt12m_3712952L,Number of clients that have used the same mobi...,numerical,double


## 6. Look at real values and missing values

We load only `case_id` and the ten selected features. Parquet can read selected columns, so this uses much less memory than loading all 168 columns. `NaN` means a value is missing.

In [7]:
columns_to_read = ["case_id"] + selected_features
selected_data = pd.concat(
    [pd.read_parquet(path, columns=columns_to_read) for path in static_paths],
    ignore_index=True,
)

print(f"Selected data: {selected_data.shape[0]:,} rows × {selected_data.shape[1]} columns")
print(f"Approximate memory: {selected_data.memory_usage(deep=True).sum() / 1024**2:,.1f} MB")
selected_data.sample(10, random_state=42)

Selected data: 1,526,659 rows × 11 columns
Approximate memory: 128.1 MB


,case_id,credamount_770A,annuity_780A,maininc_215A,applications30d_658L,numactivecreds_622L,totaldebt_9A,avgdbddpdlast24m_3658932P,avgdpdtolclosure24_3658938P,clientscnt12m_3712952L,price_1097A
1322328,1795150,19674.4,3667.8000,32000.0,0.0,0.0,0.0,NaN,0.0,0.0,19674.4
597755,1407577,31840.0,6402.0000,38000.0,0.0,1.0,9353.4,188.0,189.0,0.0,51840.0
1325416,1798238,15998.0,1631.8000,100000.0,0.0,1.0,23095.0,0.0,0.0,0.0,15998.0
877847,1687670,54000.0,4122.8003,70000.0,0.0,2.0,20393.4,-12.0,0.0,0.0,54000.0
210557,685605,9998.0,833.2000,NaN,0.0,0.0,0.0,NaN,NaN,0.0,9998.0
522239,1332061,109470.0,11168.8000,24000.0,0.0,0.0,0.0,NaN,2.0,0.0,109470.0
1214334,1022215,13998.0,1381.4000,NaN,0.0,0.0,0.0,NaN,NaN,0.0,13998.0
200141,675189,11210.0,1395.0000,NaN,0.0,0.0,0.0,NaN,NaN,0.0,11210.0
1003401,2650707,400000.0,19963.0000,60000.0,0.0,1.0,0.0,-2.0,0.0,0.0,0.0
782552,1592374,26967.0,2750.8000,NaN,0.0,0.0,0.0,NaN,NaN,0.0,26967.0


In [8]:
selected_summary = pd.DataFrame({
    "meaning": [description_map[feature] for feature in selected_features],
    "missing_count": selected_data[selected_features].isna().sum(),
    "missing_percent": selected_data[selected_features].isna().mean().mul(100),
    "minimum": selected_data[selected_features].min(),
    "median": selected_data[selected_features].median(),
    "maximum": selected_data[selected_features].max(),
})
selected_summary.index.name = "feature"
selected_summary.style.format({
    "missing_count": "{:,}",
    "missing_percent": "{:.2f}%",
    "minimum": "{:,.2f}",
    "median": "{:,.2f}",
    "maximum": "{:,.2f}",
})

,meaning,missing_count,missing_percent,minimum,median,maximum
feature,,,,,,
credamount_770A,Loan amount or credit card limit.,0,0.00%,"2,000.00","35,199.40","950,000.00"
annuity_780A,Monthly annuity amount.,0,0.00%,80.80,"3,151.80","106,007.00"
maininc_215A,Client's primary income amount.,"511,255",33.49%,0.00,"40,000.00","200,000.00"
applications30d_658L,Number of applications made by the client in the last 30 days.,0,0.00%,0.00,0.00,28.00
numactivecreds_622L,Number of active credits.,0,0.00%,0.00,0.00,7.00
totaldebt_9A,Total amount of debt.,3,0.00%,0.00,0.00,"1,210,629.10"
avgdbddpdlast24m_3658932P,Average days past or before due of payment during the last 24 months.,"613,202",40.17%,"-1,220.00",-5.00,"4,750.00"
avgdpdtolclosure24_3658938P,"Average DPD (days past due) with tolerance within the past 24 months from the maximum closure date, assuming that the contract is finished. If the contract is ongoing, the calculation is based on the current date.","467,175",30.60%,0.00,0.00,"4,750.00"
clientscnt12m_3712952L,Number of clients that have used the same mobile phone as the applicant in the past 12 months.,0,0.00%,0.00,0.00,198.00


## 7. Does every application have exactly one static row?

We read only the `case_id` column from the base table. We then verify uniqueness and compare the two ID sets. Never assume that row order matches between different tables; always join using `case_id`.

In [9]:
base_ids = pd.read_parquet(base_path, columns=["case_id"])["case_id"]
static_ids = selected_data["case_id"]

print(f"Base applications: {len(base_ids):,}")
print(f"Combined static rows: {len(static_ids):,}")
print(f"Duplicate case_id values in static_0: {static_ids.duplicated().sum():,}")
print(f"Base applications without a static row: {(~base_ids.isin(static_ids)).sum():,}")
print(f"Static rows not found in base: {(~static_ids.isin(base_ids)).sum():,}")

Base applications: 1,526,659
Combined static rows: 1,526,659
Duplicate case_id values in static_0: 0
Base applications without a static row: 0
Static rows not found in base: 0


## 8. Do we need all 167 features?

No. They are **candidate features**, not mandatory features. Our first baseline can use the ten understandable numerical columns. Later we can add other columns and retain them only when time-aware validation shows a reliable improvement.

Different columns require different preparation:

- numerical columns may need missing-value handling and scaling for logistic regression;
- categorical columns require encoding or a model with categorical support;
- date columns should usually become application-relative durations;
- constant or unreliable columns may be excluded; and
- highly related columns may be redundant.

`case_id` is only a join key and must not be used as a predictor. We also never add `target` to the feature list because it is the answer.

## What we learned

- `static_0` is one logical depth-0 table stored in two row-part files.
- It contains **1,526,659 rows and 168 columns**.
- The columns are `case_id` plus **167 candidate predictors**.
- Each application has exactly one matching `static_0` row.
- All 167 predictors have a supplied description in `feature_definitions.csv`.
- We can explore the table with pandas and Parquet metadata; Excel is unnecessary.
- We do not need every feature in the first model.

Before moving forward, use `search_features()` with several words and use the ten-feature summary to explain, in your own words, what information this table provides about an application.